# M4: ΔB across the merge spectra (PLAN §7 M4; D-081, D-082)

**Notebook settings:** Accelerator **GPU T4 x2** · Internet **on** · Secrets: **HF_TOKEN**. Run as **Save Version → Save & Run All**.

Set `CONFIGS` in the first code cell. Three sessions keep `/kaggle/working` under 20 GB:
- **Session A:** `["smoke", "tier2_qwen2.5-0.5b"]`. `smoke` is DC-11: the whole pipeline through `deltab` in < 15 min, with 24 primary-cell rows.
- **Session B:** `["tier2_llama3.2-1b", "tier1_llama3.1-8b"]`. Llama-3.1-8B also runs DC-12: `--only-ckpt a050` first, then load + extraction ≤ 15 min.
- **Session C:** `["tier1_mistral-7b", "tier1_gemma3-4b"]`.

Per config, the notebook:
1. restores the trained endpoints (`train/<slug>`) from the private store;
2. runs `sentences,ftdata,train,embed,deltab`. Training is skipped because every endpoint has `done.json`. The `sentences` stage downloads Alpaca and streams part of Tulu for the anchor pools;
3. re-runs `embed --force embed` for DC-10, where every checkpoint must be a cache hit with no model load, and then checks that `deltab` stays a cache hit;
4. syncs the embeddings, manifests and results to the private store.

Only aggregates are printed. No sentence text is printed.


In [ ]:
REPO = "SGupta-4/Reference-Based-Bias-Detection-Implementation"
REF = "main"  # a branch, a tag, or a FULL 40-character commit SHA (short SHAs fail)
WORK = "/kaggle/working/rbbd"
CONFIGS = ["smoke", "tier2_qwen2.5-0.5b"]  # session A; B: tier2_llama3.2-1b, tier1_llama3.1-8b; C: tier1_mistral-7b, tier1_gemma3-4b
DC12_CONFIGS = {"tier1_llama3.1-8b"}       # DC-12 timing probe (--only-ckpt a050) before the sweep
import time
SESSION_T0 = time.time()


In [ ]:
# Clone the repo at REF (branch, tag or commit SHA). If the repo is private, attach the
# GITHUB_TOKEN secret: it is passed to git as an HTTP header through environment
# variables, never in the URL, argv or output.
import base64, datetime, os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

os.environ.setdefault(
    "RBBD_SESSION_ID", datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
RBBD_SESSION_ID = os.environ["RBBD_SESSION_ID"]  # also expands $RBBD_SESSION_ID in ! lines
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # fail instead of prompting for a password
try:
    _gh = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    _gh = None
if _gh:
    _basic = base64.b64encode(f"x-access-token:{_gh}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {_basic}")
del _gh

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
for args in (["init", "-q"], ["remote", "add", "origin", f"https://github.com/{REPO}.git"],
             ["fetch", "-q", "--depth", "1", "origin", REF], ["checkout", "-q", "FETCH_HEAD"]):
    done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode and args[0] == "fetch" and "GIT_CONFIG_COUNT" in git_env:
        # An expired or revoked token gets HTTP 401 even on a public repo (B-025): retry
        # without it. git's stderr never contains the header value.
        print("GITHUB_TOKEN was rejected (expired or revoked?); retrying without it")
        git_env = {k: v for k, v in git_env.items() if not k.startswith("GIT_CONFIG_")}
        done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {args[0]} failed: {done.stderr[-500:]}")
del git_env
os.chdir(WORK)
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
      "| session", os.environ["RBBD_SESSION_ID"])

In [ ]:
!pip install -q -e ".[train,dev]" 2>&1 | tail -5
!pip freeze > /kaggle/working/pip_freeze_m4.txt
# Preflight (B-012): fails here, not inside a stage, if torch and torchvision do not match.
!python -c "import torch, torchvision, transformers; from transformers import Qwen2ForCausalLM, LlamaForCausalLM, Gemma3ForConditionalGeneration; print('preflight ok', torch.__version__, torchvision.__version__, transformers.__version__)"

In [ ]:
# Copy Kaggle Secrets into the environment (presence flags only are printed) and keep the
# HF cache on ephemeral disk so it never counts against /kaggle/working (D-029/D-041).
import sys
sys.path.insert(0, f"{WORK}/src")  # the editable install is not visible to this already-running kernel
from rbbd.utils.env import ephemeral_dir, load_kaggle_secrets

_secrets = load_kaggle_secrets(["HF_TOKEN"])
print(_secrets)
if not _secrets["HF_TOKEN"]:
    # Stop here instead of failing later on every gated download (B-022).
    raise RuntimeError("HF_TOKEN secret not attached: Add-ons -> Secrets -> enable HF_TOKEN "
                       "for this notebook, then run again")
os.environ["HF_HOME"] = str(ephemeral_dir() / "hf")
ENV_DIR = f"/kaggle/working/artifacts/env/{os.environ['RBBD_SESSION_ID']}"
os.makedirs(ENV_DIR, exist_ok=True)

In [ ]:
# One config at a time: restore, sweep, DC-10 re-run, sync. Aggregates only are printed.
import glob, json, shutil, subprocess
from pathlib import Path
from rbbd import config as _config, runner
from rbbd.utils import manifest as mf
ROOT = Path("/kaggle/working/artifacts")
SUMMARY = {}

def manifest(cfg, stage):
    return mf.read(mf.manifest_path(ROOT, stage, runner.stage_run_keys(cfg)[stage]))

def stage_file(cfg, stage, name):
    return ROOT / stage / runner.stage_run_keys(cfg)[stage] / name

def show_errors(slug):
    for log in sorted(glob.glob(f"{ROOT}/train/{slug}/*/*/*/*/train.log")):
        if "Error" in open(log).read():
            print("----", log.split("/train/")[1]); print("".join(open(log).readlines()[-15:]))

for NAME in CONFIGS:
    CFG = f"configs/{NAME}.yaml"
    cfg = _config.load(CFG)
    SLUG, RUN = cfg["models"][0]["slug"], cfg["run_name"]
    out = SUMMARY.setdefault(NAME, {"config_hash": cfg.hash})
    print(f"===== {NAME} ({SLUG}) =====")
    if NAME != "smoke":
        !python -m rbbd.cli restore --path train/{SLUG}
    !python -m rbbd.cli run --config {CFG} --stages sentences,ftdata
    if NAME != "smoke":
        # Never train here: every endpoint must already be done (a changed dataset revision
        # would give new train keys and silently start hours of training).
        from rbbd.finetune import sft
        jobs = sft.plan_jobs(cfg, ROOT, list(manifest(cfg, "ftdata").output_hashes))
        missing = [str(j.out_dir) for j in jobs if not (j.out_dir / "done.json").exists()]
        if missing:
            raise RuntimeError(f"{NAME}: endpoints not found after restore (train keys changed?): {missing}")
    if NAME in DC12_CONFIGS:
        !python -m rbbd.cli run --config {CFG} --stages train
        !python -m rbbd.cli run --config {CFG} --stages embed --only-ckpt a050
        t = json.loads(stage_file(cfg, "embed", "timing_only_a050.json").read_text())["models"][SLUG]
        ext = {k: v.get("seconds") for k, v in t["checkpoints"].items()}
        out["dc12"] = {"load_seconds": t["load_seconds"], "download_seconds": t["download_seconds"],
                       "extraction_seconds": ext,
                       "load_plus_extraction_s": (t["load_seconds"] or 0) + max(v or 0 for v in ext.values())}
        shutil.copy(stage_file(cfg, "embed", "timing_only_a050.json"), f"{ENV_DIR}/dc12_{NAME}.json")
    t0 = time.time()
    !python -m rbbd.cli run --config {CFG} --stages sentences,ftdata,train,embed,deltab
    out["pipeline_wall_s"] = round(time.time() - t0, 1)
    emb_m, dlt_m = manifest(cfg, "embed"), manifest(cfg, "deltab")
    out["embed_complete"] = bool(emb_m and emb_m.complete)
    out["deltab_complete"] = bool(dlt_m and dlt_m.complete)
    if not (out["embed_complete"] and out["deltab_complete"]):
        show_errors(SLUG); print(json.dumps(out, indent=1)); continue
    timing = json.loads(stage_file(cfg, "embed", "timing.json").read_text())
    shutil.copy(stage_file(cfg, "embed", "timing.json"), f"{ENV_DIR}/embed_timing_{NAME}.json")
    m = timing["models"][SLUG]
    out["embed"] = {"n_texts": timing["n_texts"], "load_seconds": m["load_seconds"],
                    "checkpoints": len(m["checkpoints"]),
                    "extraction_s_total": round(sum(v.get("seconds", 0) for v in m["checkpoints"].values()), 1),
                    "peak_gib_per_gpu": {d: max((v.get("peak_mem_gib") or {}).get(d, 0) for v in m["checkpoints"].values())
                                         for d in {d for v in m["checkpoints"].values() for d in (v.get("peak_mem_gib") or {})}}}
    union = json.loads(next((ROOT / p) for p in manifest(cfg, "sentences").output_hashes if p.endswith("union.json")).read_text())
    pools = union["stats"].get("anchor_pools", {})
    out["anchor_pools"] = {k: {kk: vv for kk, vv in v.items() if kk != "candidates_by_source"} if isinstance(v, dict) else v
                           for k, v in pools.items()}
    out["tulu_sources"] = pools.get("tulu", {}).get("candidates_by_source")
    out["union_hash"] = union["union_hash"]
    dtiming = json.loads(stage_file(cfg, "deltab", "timing.json").read_text())
    out["deltab"] = {"seconds": dtiming["seconds"], "rows": dtiming["rows"]}
    out["sanity"] = json.loads(stage_file(cfg, "deltab", "sanity.json").read_text())
    shutil.copy(stage_file(cfg, "deltab", "timing.json"), f"{ENV_DIR}/deltab_timing_{NAME}.json")
    # DC-10: the same sweep again; every checkpoint must hit the cache and nothing may load.
    !python -m rbbd.cli run --config {CFG} --stages embed --force embed
    again = json.loads(stage_file(cfg, "embed", "timing.json").read_text())["models"][SLUG]
    out["dc10"] = {"all_cache_hits": all(v["cache_hit"] for v in again["checkpoints"].values()),
                   "checkpoints": len(again["checkpoints"]), "model_loaded": again["load_seconds"] is not None}
    r = subprocess.run([sys.executable, "-m", "rbbd.cli", "run", "--config", CFG, "--stages", "deltab"],
                       capture_output=True, text=True)
    out["dc10"]["deltab_cache_hit"] = "cache hit: stage deltab" in (r.stdout + r.stderr)
    # Results (aggregates) into the artifact tree so they sync with everything else.
    res = Path(WORK) / "results" / RUN
    if res.is_dir():
        shutil.copytree(res, ROOT / "results" / RUN, dirs_exist_ok=True)
    rows = [l for l in open(res / "delta_b.csv")][1:]
    out["primary_rows"] = len(rows)
    for path in ["manifests", "sentences", "ftdata", "embed", "deltab", f"embeddings/{SLUG}", f"results/{RUN}"]:
        if (ROOT / path).exists():
            !python -m rbbd.cli sync --path {path}
    print(json.dumps(out, indent=1, default=str))


In [ ]:
# Summary to paste back (aggregates only).
!python -m rbbd.cli sync --path env/$RBBD_SESSION_ID
print(json.dumps(SUMMARY, indent=1, default=str))
print("session hours:", round((time.time() - SESSION_T0) / 3600, 2))
!df -h /kaggle/working | tail -1
for NAME in CONFIGS:
    !python -m rbbd.cli status --config configs/{NAME}.yaml
